In [ ]:
# ── Loop over all 3D Static scenes ────────────────────────────────────────────
import os, subprocess

Path         = f'/home/daniel/Documents/Projects/'
Project_Path = Path + f'Sync/3D/'
Train_Path   = Project_Path + f'3DSQS/train_joint.py'
Render_Path  = Project_Path + f'3DSQS/render_by_interp.py'

Splat_Type_3D = "SQE"
iter_3D       = 30_000
max_splats_3D = 10_000_000

# Scenes: (name, resolution, images_subfolder)
#   resolution       — integer divisor applied to image size during training
#   images_subfolder — passed as --images; None means default (images/)
#
# Resolution chosen so longest axis < 1000 px after downsampling.
Datasets_3D = {
    "Deep_Blending": {
        "scenes": [
            # native → downsampled (longest axis)
            ("Aquarium-20",    2, None),   # 2633 → 658
            ("Bedroom",        1, None),   # 1297 → 648
            ("Boats",          4, None),   # 3674 → 918
            ("Bridge",         1, None),   # 1440 → 720
            ("CreepyAttic",    1, None),   # 1236 → 618
            ("DrJohnson",      1, None),   # 1333 → 666
            ("Hugo-1",         4, None),   # 3210 → 802
            ("Library",        4, None),   # 3569 → 892
            ("Museum-1",       2, None),   # 2354 → 588
            ("Museum-2",       2, None),   # 2351 → 587
            ("Playroom",       1, None),   # 1264 → 632
            ("Ponche",         2, None),   # 2508 → 627
            ("SaintAnne",      2, None),   # 2504 → 626
            ("Shed",           4, None),   # 3672 → 918
            ("Tree-18",        2, None),   # 2666 → 666
            ("Yellowhouse-12", 4, None),   # 4248 → 531  (÷4=1062 exceeds 1000)
        ],
        "path": lambda s: f'{Path}Datasets/Static/deep_blending/{s}/colmap/',
    },
    "Mip_nerf_360": {
        "scenes": [
            # pre-downsampled folder chosen so longest axis < 1000 px
            ("bicycle",  1, "images_4"),   # 618
            ("bonsai",   1, "images_4"),   # 780
            ("counter",  1, "images_4"),   # 779
            ("flowers",  1, "images_4"),   # 628
            ("garden",   1, "images_4"),   # 648
            ("kitchen",  1, "images_4"),   # 779
            ("room",     1, "images_4"),   # 779
            ("stump",    1, "images_4"),   # 622
            ("treehill", 1, "images_4"),   # 634
        ],
        "path": lambda s: f'{Path}Datasets/Static/mip_nerf_360/{s}/',
    },
    "Tanks_Temples": {
        "scenes": [
            # native → downsampled (longest axis)
            ("Family",     2, None),   # 1920 → 960
            ("Francis",    2, None),   # 1920 → 960
            ("Horse",      2, None),   # 1920 → 960
            ("Lighthouse", 2, None),   # 2048 → 512  (÷2=1024 exceeds 1000)
            ("M60",        2, None),   # 2048 → 512  (÷2=1024 exceeds 1000)
            ("Panther",    2, None),   # 2048 → 512  (÷2=1024 exceeds 1000)
            ("Playground", 2, None),   # 1920 → 960
            ("Train",      2, None),   # 1920 → 960
        ],
        "path": lambda s: f'{Path}Datasets/Static/tanks_temples/intermediate/{s}/',
    },
}

python = '/home/daniel/anaconda3/envs/3DSQS/bin/python'
failures = []  # collect (label, reason) for end-of-run summary

def run_cmd(label, args):
    """Run a subprocess, inheriting stdout/stderr so output streams live to the notebook.
    Returns True on success, False on failure."""
    print(f"\n{'='*66}")
    print(f"[{label}]")
    print(f"{'='*66}")
    try:
        # No stdout/stderr redirection — subprocess inherits the kernel's file
        # descriptors so output streams directly to the notebook cell output.
        result = subprocess.run(args)
        if result.returncode != 0:
            reason = f"exit code {result.returncode}"
            print(f"\n[FAILED] {label} — {reason}")
            failures.append((label, reason))
            return False
        return True
    except KeyboardInterrupt:
        reason = "KeyboardInterrupt"
        print(f"\n[INTERRUPTED] {label} — skipping to next scene (Ctrl+C again to stop all)")
        failures.append((label, reason))
        return False

# ── Scene summary ──────────────────────────────────────────────────────────────
print("{:<20} {:<20} {:>10} {:<12}".format("Dataset", "Scene", "resolution", "images"))
print('-' * 66)
for _ds, _cfg in Datasets_3D.items():
    for _scene, _res, _img in _cfg['scenes']:
        print(f"{_ds:<20} {_scene:<20} {_res:>10} {str(_img):<12}")
print()

for Dataset, cfg in Datasets_3D.items():
    for Scene, resolution, images in cfg["scenes"]:

        scene_path  = cfg["path"](Scene)
        output_path = f'{Project_Path}Results/{Dataset}/{Scene}/{Splat_Type_3D}'
        label       = f"{Dataset}/{Scene}"

        if not os.path.exists(scene_path):
            print(f"[SKIP] {label} — not found: {scene_path}")
            failures.append((label, "scene path not found"))
            continue

        os.makedirs(output_path, exist_ok=True)

        # ── Training ──────────────────────────────────────────────────────────
        train_args = [
            python, Train_Path,
            "-s", scene_path, "-m", output_path,
            "--scene", Scene, "--iter", str(iter_3D),
            "--optim_pose", "--results", output_path,
            "--splat_type", Splat_Type_3D, "--resolution", str(resolution),
            "--dtype", "fp32", "--max_splats", str(max_splats_3D),
            "--step", "0", "--device", "cuda",
        ]
        if images:
            train_args += ["--images", images]

        if not run_cmd(f"train  {label}", train_args):
            continue

        # ── Render fly-through video ──────────────────────────────────────────
        if not os.path.exists(f"{output_path}/model.ply"):
            msg = "no model.ply after training"
            print(f"[SKIP render] {label} — {msg}")
            failures.append((label, msg))
            continue

        render_args = [
            python, Render_Path,
            "-s", scene_path, "-m", output_path,
            "--scene", Scene, "--iter", str(iter_3D),
            "--eval", "--get_video", "--results", output_path,
            "--splat_type", Splat_Type_3D, "--resolution", str(resolution),
            "--device", "cuda",
        ]
        if images:
            render_args += ["--images", images]

        run_cmd(f"render {label}", render_args)

# ── End-of-run failure summary ─────────────────────────────────────────────────
if failures:
    print("\n" + "="*66)
    print(f"FAILURES ({len(failures)}):")
    for lbl, reason in failures:
        print(f"  {lbl:<40} {reason}")
else:
    print("\nAll scenes completed successfully.")

Dataset              Scene                resolution images      
------------------------------------------------------------------
Deep_Blending        Aquarium-20                   2 None        
Deep_Blending        Bedroom                       1 None        
Deep_Blending        Boats                         4 None        
Deep_Blending        Bridge                        1 None        
Deep_Blending        CreepyAttic                   1 None        
Deep_Blending        DrJohnson                     1 None        
Deep_Blending        Hugo-1                        4 None        
Deep_Blending        Library                       4 None        
Deep_Blending        Museum-1                      2 None        
Deep_Blending        Museum-2                      2 None        
Deep_Blending        Playroom                      1 None        
Deep_Blending        Ponche                        2 None        
Deep_Blending        SaintAnne                     2 None        
Deep_Blen

/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
Optimizing /home/daniel/Documents/Projects/Sync/3D/Results/Deep_Blending/Aquarium-20/SQE
Output folder: /home/daniel/Documents/Projects/Sync/3D/Results/Deep_Blending/Aquarium-20/SQE
Init: COLMAP  (/home/daniel/Documents/Projects/Datasets/Static/deep_blending/Aquarium-20/colmap)
/home/daniel/Documents/Projects/Datasets/Static/deep_blending/Aquarium-20/colmap/sparse True
Reading camera 19/19

/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/accelerate/accelerator.py:528: UserWarning: `log_with=[]` was passed but no supported trackers are currently installed.
  warnings.warn(f"`log_with={log_with}` was passed but no supported trackers are currently installed.")
  0%|          | 0/30000 [00:00<?, ?it/s]


  Train cameras: 16,  Test cameras (1-in-8 holdout): 3
Loading Training Cameras
train_camera_num:  16
Loading Test Cameras
test_camera_num:  3
Number of points at initialisation :  19121
Image size: 1312×976  (16 train cameras)


...

/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
Optimizing /home/daniel/Documents/Projects/Sync/3D/Results/Tanks_Temples/Lighthouse/SQE
Output folder: /home/daniel/Documents/Projects/Sync/3D/Results/Tanks_Temples/Lighthouse/SQE
Init: COLMAP  (/home/daniel/Documents/Projects/Datasets/Static/tanks_temples/intermediate/Lighthouse)
/home/daniel/Documents/Projects/Datasets/Static/tanks_temples/intermediate/Lighthouse/sparse True
Reading camera 309/309

/home/daniel/anaconda3/envs/3DSQS/lib/python3.10/site-packages/accelerate/accelerator.py:528: UserWarning: `log_with=[]` was passed but no supported trackers are currently installed.
  warnings.warn(f"`log_with={log_with}` was passed but no supported trackers are currently installed.")
  0%|          | 0/30000 [00:00<?, ?it/s]


  Train cameras: 270,  Test cameras (1-in-8 holdout): 39
Loading Training Cameras
train_camera_num:  270
Loading Test Cameras
test_camera_num:  39
Number of points at initialisation :  179414
Image size: 1024×544  (270 train cameras)


loss: 0.171 total: 0.171 l1: 0.136 ssim: 0.687 psnr: 15.144:   4%|▍         | 1180/30000 [01:55<48:53,  9.82it/s]  

In [4]:
Dataset = "Sora"
Scenes = [["Santorini", 3]]

Dataset = "Deep_Blending"
Scenes = [["Aquarium-20", 19, 4], ["Bedroom", 30, 2], ["Boats", 30, 8], ["Bridge", 30, 2], ["CreepyAttic", 30, 2], ["DrJohnson", 24, 2], ["Hugo-1", 24, 2], ["Library", 30, 8],
              ["Lumber", 30, 8], ["Museum-1", 27, 4], ["Museum-2", 30, 4], ["NightSnow", 30, 4], ["Playroom", 30, 2], ["Ponche", 30, 4], ["SaintAnne", 30, 4], ["Shed", 30, 8],
              ["Street-10", 13, 4], ["Tree-18", 18, 4], ["Yellowhouse-12", 12, 2]]

# Dataset = "Tanks_Temples/intermediate"
# Scenes = [["Family", 30, 4], ["Francis", 30, 4], ["Horse", 30, 4], ["Lighthouse", 30, 4], ["M60", 30, 4], ["Panther", 30, 4], ["Playground", 30, 4], ["Train", 30, 4]]

# Dataset = "Mip_nerf_360/"
# Scenes = [["Bicycle", 30, 1], ["Bonsai", 30, 1], ["Counter", 30, 1], ["Flowers", 30, 1], ["Garden", 30, 1], ["Kitchen", 30, 1], ["Room", 30, 1], ["Stump", 30, 1], ["Treehill", 30, 1]]

# Dataset = "Custom/"
# Scenes = [["Rock", 32, 2]]



Scene, n_views, resolution = Scenes[12]
Splat_Type = "SQE"
iter       = 10000
dtype      = "fp32"
max_splats = 1_000_000
step       = 0
init_type  = "colmap"   # "colmap" (default) or "dust3r" (run Init cell first)
# init_type  = "dust3r"   # "colmap" (default) or "dust3r" (run Init cell first)


Path = f'/home/daniel/Documents/Projects/'
Project_Path = Path + f'Sync/3D/'

Dataset_Path = Path + f'Datasets/Static/{Dataset.lower()}/'
Scene_Path = Dataset_Path + Scene + f'/colmap/'

Model_Path = Project_Path + "3DSQS/submodules/dust3r/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth"

Init_Path = Project_Path + f"3DSQS/coarse_init_infer.py"
Train_Path = Project_Path + f"3DSQS/train_joint.py"
Render_Path = Project_Path + f"3DSQS/render_by_interp.py"

Output_Path  = Project_Path + f'Results/{Dataset}/{Scene}/{Splat_Type}'
result_path  = Output_Path   # alias used by the render cell below



import torch as tc
device = tc.device('cuda')
tc.cuda.empty_cache()
total_mem = round(tc.cuda.get_device_properties(0).total_memory / 2**30, 1)
print(Scene, n_views, Splat_Type, max_splats, step, total_mem, device)

print(Dataset_Path, '\n', Scene_Path, '\n', Output_Path)

# print(torch.cuda.memory_summary(device=None, abbreviated=False))

Playroom 30 SQE 1000000 0 23.6 cuda
/home/daniel/Documents/Projects/Datasets/Static/deep_blending/ 
 /home/daniel/Documents/Projects/Datasets/Static/deep_blending/Playroom/colmap/ 
 /home/daniel/Documents/Projects/Sync/3D/Results/Deep_Blending/Playroom/SQE


In [ ]:
cmd = f"""
python "{Train_Path}" \
    -s "{Scene_Path}" \
    -m "{Output_Path}" \
    --scene "{Scene}" \
    --n_views "{n_views}" \
    --iter "{iter}" \
    --optim_pose \
    --results "{Output_Path}" \
    --splat_type "{Splat_Type}" \
    --resolution "{resolution}" \
    --dtype "{dtype}" \
    --max_splats "{max_splats}" \
    --step "{step}" \
    --device "{device}" \
    --init_type "{init_type}" \
"""
print(cmd)
!{cmd}

In [ ]:
%run {Render_Path} -s {Scene_Path} -m {Output_Path} --n_views {n_views} --scene {Scene} --iter {iter} --eval --get_video --results {Output_Path} --splat_type {Splat_Type} --resolution {resolution} --device {device}


In [ ]:
%run {Init_Path} --n_views {n_views} --img_base_path {Scene_Path} --model_path {Model_Path}

In [ ]:
import importlib
import diff_superquadric_rasterization
importlib.reload(diff_superquadric_rasterization)

%run '/home/daniel/Documents/Projects/Sync/3D/3DSQS/render_test.py'

In [3]:
import sys
print(sys.executable)

/home/daniel/anaconda3/envs/3DSQS/bin/python


In [ ]:
torch.cuda.memory._dump_snapshot("Memory.pickle")

In [8]:
# !conda activate 3DSQS
# !python -m ipykernel install --user --name 3DSQS --display-name "Python (3DSQS)"


CondaError: Run 'conda init' before 'conda activate'

Installed kernelspec 3DSQS in /home/daniel/.local/share/jupyter/kernels/3dsqs
